In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [10]:
!pip install mauve-text -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 99.1 MB/s eta 0:00:00


In [2]:
# Vanilla and Vanilla CoT NLI evaluation of recall and precision
import json
import os
import re
import copy
import numpy as np
from tqdm import tqdm
from nltk.tokenize import sent_tokenize
from transformers import pipeline
import nltk
nltk.download('punkt_tab')

# -----------------------------
# CONFIG
# -----------------------------
RESULTS_DIR = "/content/drive/MyDrive/results2/eli5_tot"
N_EXAMPLES = 5  # how many good/bad examples to save per file

# -----------------------------
# LOAD NLI MODEL
# -----------------------------
nli = pipeline(
    "text-classification",
    model="facebook/bart-large-mnli",
    device=-1,
    truncation=True,
    max_length=1024,
)

# -----------------------------
# HELPERS
# -----------------------------
def run_nli(premise, hypothesis, threshold=0.5):
    result = nli(f"{premise} {hypothesis}")[0]
    return 1 if (result["label"].lower() == "entailment" and result["score"] >= threshold) else 0


def extract_citations(sentence):
    """Extract [1][2] → [0,1]"""
    refs = re.findall(r"\[(\d+)\]", sentence)
    return [int(r) - 1 for r in refs]


def remove_citations(text):
    return re.sub(r"\[\d+\]", "", text).strip()


def format_doc(doc):
    return f"{doc.get('title', '')}\n{doc.get('text', '')}"


# -----------------------------
# MAIN EVAL FUNCTION
# -----------------------------
def evaluate_file(data):

    recall_scores = []
    precision_scores = []
    per_item_scores = []  # track per-item for example extraction

    for item in tqdm(data):
        output = item["output"]

        if not output or not output.strip():
          continue

        docs = item.get("docs", [])

        sentences = sent_tokenize(output)

        sent_recalls = []
        sent_precisions = []

        for sent in sentences:
            clean_sent = remove_citations(sent)
            refs = extract_citations(sent)

            # -----------------------------
            # RECALL
            # -----------------------------
            if len(refs) == 0 or any(r >= len(docs) for r in refs):
                sent_recalls.append(0)
                sent_precisions.append(0)
                continue

            joint_passage = "\n".join([format_doc(docs[r]) for r in refs])

            recall = run_nli(joint_passage, clean_sent)
            sent_recalls.append(recall)

            # -----------------------------
            # PRECISION
            # -----------------------------
            if recall == 0:
                sent_precisions.append(0)
                continue

            correct_citations = 0

            for r in refs:
                # Check if single doc supports
                single_passage = format_doc(docs[r])
                single_entail = run_nli(single_passage, clean_sent)

                if single_entail:
                    correct_citations += 1
                    continue

                # Check if removing it still works
                subset = copy.deepcopy(refs)
                subset.remove(r)

                if len(subset) == 0:
                    continue

                subset_passage = "\n".join([format_doc(docs[s]) for s in subset])
                subset_entail = run_nli(subset_passage, clean_sent)

                if not subset_entail:
                    # necessary
                    correct_citations += 1
                # else: irrelevant → do not count

            precision = correct_citations / len(refs)
            sent_precisions.append(precision)

        # Aggregate per example
        if len(sentences) > 0:
            item_recall = float(np.mean(sent_recalls))
            item_precision = float(np.mean(sent_precisions))
            recall_scores.append(item_recall)
            precision_scores.append(item_precision)
            per_item_scores.append({
                "question": item.get("question", ""),
                "answer": item.get("answer", ""),
                "output": output,
                "docs": [{"title": d.get("title", ""), "text": d.get("text", "")} for d in docs],
                "citation_recall": item_recall,
                "citation_precision": item_precision,
            })

    # Sort by recall for good/bad examples
    sorted_by_recall = sorted(per_item_scores, key=lambda x: x["citation_recall"])
    bad_examples = sorted_by_recall[:N_EXAMPLES]
    good_examples = sorted_by_recall[-N_EXAMPLES:][::-1]

    return {
        "citation_recall": 100 * np.mean(recall_scores),
        "citation_precision": 100 * np.mean(precision_scores),
        "good_examples": good_examples,
        "bad_examples": bad_examples,
    }


# -----------------------------
# RUN OVER FOLDER
# -----------------------------
def main():
    results = {}
    examples_out = {}

    for filename in os.listdir(RESULTS_DIR):
        if not filename.endswith(".json"):
            continue

        path = os.path.join(RESULTS_DIR, filename)

        with open(path) as f:
            raw = json.load(f)
            data = raw["data"] if isinstance(raw, dict) else raw

        print(f"\nEvaluating {filename}...")

        scores = evaluate_file(data)

        # Separate scores from examples for the scores file
        results[filename] = {
            "citation_recall": scores["citation_recall"],
            "citation_precision": scores["citation_precision"],
        }
        examples_out[filename] = {
            "good_examples": scores["good_examples"],
            "bad_examples": scores["bad_examples"],
        }

        print({
            "citation_recall": scores["citation_recall"],
            "citation_precision": scores["citation_precision"],
        })

    score_path = "/content/drive/MyDrive/citation_scores_bart_eli5_tot.json"
    example_path = "/content/drive/MyDrive/citation_examples_bart_eli5_tot.json"

    with open(score_path, "w") as f:
        json.dump(results, f, indent=2)

    with open(example_path, "w") as f:
        json.dump(examples_out, f, indent=2, ensure_ascii=False)

    print(f"\nSaved scores to {score_path}")
    print(f"Saved examples to {example_path}")


if __name__ == "__main__":
    main()

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

...

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]


Evaluating eli5-gemma-4-26b-a4b-it-tot_gemma26-shot1-ndoc3-42-quick_test200.json...


100%|██████████| 200/200 [31:35<00:00,  9.48s/it]

{'citation_recall': np.float64(22.99920187807387), 'citation_precision': np.float64(21.67535642581489)}

Saved scores to /content/drive/MyDrive/citation_scores_bart_eli5_tot.json
Saved examples to /content/drive/MyDrive/citation_examples_bart_eli5_tot.json


In [16]:
# ToT NLI evaluation of recall and precision
import json
import os
import re
import copy
import numpy as np
from tqdm import tqdm
from nltk.tokenize import sent_tokenize
from transformers import pipeline
import nltk
nltk.download('punkt_tab')

# -----------------------------
# CONFIG
# -----------------------------
RESULTS_DIR = "/content/drive/MyDrive/results2/tot_analysis"
N_EXAMPLES = 5  # how many good/bad examples to save per file

# -----------------------------
# LOAD NLI MODEL
# -----------------------------
nli = pipeline(
    "text-classification",
    model="facebook/bart-large-mnli",
    device=-1,
    truncation=True,
    max_length=1024,
)

# -----------------------------
# HELPERS
# -----------------------------
def run_nli(premise, hypothesis, threshold=0.5):
    result = nli(f"{premise} {hypothesis}")[0]
    return 1 if (result["label"].lower() == "entailment" and result["score"] >= threshold) else 0


def extract_citations(sentence):
    """Extract [1][2] → [0,1]"""
    refs = re.findall(r"\[(\d+)\]", sentence)
    return [int(r) - 1 for r in refs]


def remove_citations(text):
    return re.sub(r"\[\d+\]", "", text).strip()


def format_doc(doc):
    return f"{doc.get('title', '')}\n{doc.get('text', '')}"

def extract_final_answer(output):
    """Extract only the Final Answer paragraph from ToT output."""
    match = re.search(r"Final Answer[:\s]+(.*)", output, re.DOTALL | re.IGNORECASE)
    if match:
        return match.group(1).strip()
    return output  # fallback to full output if not found

# -----------------------------
# MAIN EVAL FUNCTION
# -----------------------------
def evaluate_file(data):

    recall_scores = []
    precision_scores = []
    per_item_scores = []  # track per-item for example extraction

    for item in tqdm(data):
        output = extract_final_answer(item["output"])

        if not output or not output.strip():
          continue

        docs = item.get("docs", [])

        sentences = sent_tokenize(output)

        sent_recalls = []
        sent_precisions = []

        for sent in sentences:
            clean_sent = remove_citations(sent)
            refs = extract_citations(sent)

            # -----------------------------
            # RECALL
            # -----------------------------
            if len(refs) == 0 or any(r >= len(docs) for r in refs):
                sent_recalls.append(0)
                sent_precisions.append(0)
                continue

            joint_passage = "\n".join([format_doc(docs[r]) for r in refs])

            recall = run_nli(joint_passage, clean_sent)
            sent_recalls.append(recall)

            # -----------------------------
            # PRECISION
            # -----------------------------
            if recall == 0:
                sent_precisions.append(0)
                continue

            correct_citations = 0

            for r in refs:
                # Check if single doc supports
                single_passage = format_doc(docs[r])
                single_entail = run_nli(single_passage, clean_sent)

                if single_entail:
                    correct_citations += 1
                    continue

                # Check if removing it still works
                subset = copy.deepcopy(refs)
                subset.remove(r)

                if len(subset) == 0:
                    continue

                subset_passage = "\n".join([format_doc(docs[s]) for s in subset])
                subset_entail = run_nli(subset_passage, clean_sent)

                if not subset_entail:
                    # necessary
                    correct_citations += 1
                # else: irrelevant → do not count

            precision = correct_citations / len(refs)
            sent_precisions.append(precision)

        # Aggregate per example
        if len(sentences) > 0:
            item_recall = float(np.mean(sent_recalls))
            item_precision = float(np.mean(sent_precisions))
            recall_scores.append(item_recall)
            precision_scores.append(item_precision)
            per_item_scores.append({
                "question": item.get("question", ""),
                "answer": item.get("answer", ""),
                "output": output,
                "docs": [{"title": d.get("title", ""), "text": d.get("text", "")} for d in docs],
                "citation_recall": item_recall,
                "citation_precision": item_precision,
            })

    # Sort by recall for good/bad examples
    sorted_by_recall = sorted(per_item_scores, key=lambda x: x["citation_recall"])
    bad_examples = sorted_by_recall[:N_EXAMPLES]
    good_examples = sorted_by_recall[-N_EXAMPLES:][::-1]

    return {
        "citation_recall": 100 * np.mean(recall_scores),
        "citation_precision": 100 * np.mean(precision_scores),
        "good_examples": good_examples,
        "bad_examples": bad_examples,
    }


# -----------------------------
# RUN OVER FOLDER
# -----------------------------
def main():
    results = {}
    examples_out = {}

    for filename in os.listdir(RESULTS_DIR):
        if not filename.endswith(".json"):
            continue

        path = os.path.join(RESULTS_DIR, filename)

        with open(path) as f:
            raw = json.load(f)
            data = raw["data"] if isinstance(raw, dict) else raw

        print(f"\nEvaluating {filename}...")

        scores = evaluate_file(data)

        # Separate scores from examples for the scores file
        results[filename] = {
            "citation_recall": scores["citation_recall"],
            "citation_precision": scores["citation_precision"],
        }
        examples_out[filename] = {
            "good_examples": scores["good_examples"],
            "bad_examples": scores["bad_examples"],
        }

        print({
            "citation_recall": scores["citation_recall"],
            "citation_precision": scores["citation_precision"],
        })

    score_path = "/content/drive/MyDrive/results2/citation_results/citation_scores_bart_tot.json"
    example_path = "/content/drive/MyDrive/citation_results/citation_examples_bart_tot.json"

    with open(score_path, "w") as f:
        json.dump(results, f, indent=2)

    with open(example_path, "w") as f:
        json.dump(examples_out, f, indent=2, ensure_ascii=False)

    print(f"\nSaved scores to {score_path}")
    print(f"Saved examples to {example_path}")


if __name__ == "__main__":
    main()

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]


Evaluating eli5-gemma-4-26b-a4b-it-tot_gemma26-shot1-ndoc3-42-quick_test200.json...


100%|██████████| 200/200 [07:55<00:00,  2.38s/it]


{'citation_recall': np.float64(62.066666666666656), 'citation_precision': np.float64(58.01805555555555)}

Evaluating qasa-gemma-4-26b-a4b-it-qasa_gemma26_tot-shot1-ndoc3-42-quick_test200.json...


100%|██████████| 200/200 [11:27<00:00,  3.44s/it]


{'citation_recall': np.float64(78.725), 'citation_precision': np.float64(77.28055555555555)}


FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/citation_results/citation_examples_bart_tot.json'

In [13]:
# code to save results json files as xlsx files for manual analysis!
# === 2. Imports ===
import os
import json
import pandas as pd
from glob import glob

# === 3. Path to your folder ===
folder_path = "/content/drive/MyDrive/results2/qasa_tot"

# Optional: output subfolder
output_folder = os.path.join(folder_path, "tables")
os.makedirs(output_folder, exist_ok=True)

# === 4. Collect all JSON files ===
json_files = glob(os.path.join(folder_path, "*.json"))
print(f"Found {len(json_files)} JSON files")

# === 5. Process each file separately ===
for file in json_files:
    with open(file, "r", encoding="utf-8") as f:
        data = json.load(f)

    args = data.get("args", {})
    entries = data.get("data", [])

    rows = []

    for entry in entries:
        question = entry.get("question", "")
        answer = entry.get("answer", "")
        output = entry.get("output", "")

        docs = entry.get("docs", [])
        docs_text = "\n\n".join(
            [f"[{i+1}] {d.get('title','')}: {d.get('text','')}" for i, d in enumerate(docs)]
        )

        rows.append({
            "model": args.get("model"),
            "ndoc": args.get("ndoc"),
            "shot": args.get("shot"),
            "question": question,
            "gold_answer": answer,
            "model_output": output,
            "docs": docs_text
        })

    # === 6. Create DataFrame ===
    df = pd.DataFrame(rows)

    # === 7. Save per file ===
    base_name = os.path.splitext(os.path.basename(file))[0]

    csv_path = os.path.join(output_folder, f"{base_name}.csv")
    excel_path = os.path.join(output_folder, f"{base_name}.xlsx")

    df.to_csv(csv_path, index=False)
    df.to_excel(excel_path, index=False)

    print(f"Saved: {base_name}.csv and .xlsx")

print("\nDone!")

Found 1 JSON files
Saved: qasa-gemma-4-26b-a4b-it-qasa_gemma26_tot-shot1-ndoc3-42-quick_test200.csv and .xlsx

Done!


In [12]:
import json
import os
import re
import string
import numpy as np
import mauve

from google.colab import drive
drive.mount('/content/drive')

# !pip install mauve-text -q  # uncomment if needed

# -----------------------------
# CONFIG
# -----------------------------
name = "results2/eli5"
RESULTS_DIR = f"/content/drive/MyDrive/{name}"
SCORES_OUTPUT = f"/content/drive/MyDrive/mauve_scores_eli5.json"
DEVICE_ID = 0   # set to -1 if no GPU
TRUNCATE_WORDS = 300

# -----------------------------
# MAIN EVAL FUNCTION
# -----------------------------
def evaluate_file(filename, data):
    human_data = []
    model_data = []

    for item in data:
        output = item.get("output", "")

        if not output or not output.strip():
            continue

        question = item.get("question", "")

        answer = item.get("answer", "")
        if not answer:
            texts = item.get("answers", {}).get("text", [])
            scores = item.get("answers", {}).get("score", [])
            if texts:
                best_idx = scores.index(max(scores)) if scores else 0
                answer = texts[best_idx]

        # Strip citations from both sides
        answer = re.sub(r"\[\d+\]", "", answer).strip()
        output = re.sub(r"\[\d+\]", "", output).strip()

        human_text = " ".join((question + " " + answer).split()[:TRUNCATE_WORDS]).rstrip(string.punctuation)
        model_text = " ".join((question + " " + output).split()[:TRUNCATE_WORDS]).rstrip(string.punctuation)

        human_data.append(human_text)
        model_data.append(model_text)

    if len(human_data) == 0:
        print("  [!] No valid items — skipping.")
        return None

    print(f"  n_items: {len(human_data)}")
    for i in range(min(3, len(human_data))):
        print(f"\n  --- Example {i+1} ---")
        print(f"  HUMAN: {human_data[i][:150]}")
        print(f"  MODEL: {model_data[i][:150]}")

    out = mauve.compute_mauve(
        p_text=human_data,
        q_text=model_data,
        device_id=DEVICE_ID,
        max_text_length=512,
        verbose=True,
        batch_size=8,
        featurize_model_name="gpt2-large",
    )

    return float(out.mauve * 100)


# -----------------------------
# RUN OVER FOLDER
# -----------------------------
def main():
    results = {}

    for filename in os.listdir(RESULTS_DIR):
        if not filename.endswith(".json"):
            continue

        path = os.path.join(RESULTS_DIR, filename)

        with open(path) as f:
            raw = json.load(f)
            data = raw["data"] if isinstance(raw, dict) else raw

        print(f"\nEvaluating {filename}...")

        score = evaluate_file(filename, data)

        results[filename] = {
            "mauve": round(score, 4) if score is not None else None,
        }

        print(f"  => MAUVE: {results[filename]['mauve']}")

    with open(SCORES_OUTPUT, "w") as f:
        json.dump(results, f, indent=2)

    print(f"\nSaved scores to {SCORES_OUTPUT}")
    print(json.dumps(results, indent=2))


if __name__ == "__main__":
    main()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

Evaluating eli5-gemma-4-26b-a4b-it-openrouter-shot1-ndoc3-42-quick_test200.json...
  n_items: 200

  --- Example 1 ---
  HUMAN: If Caitlin Jenner goes to prison for Vehicular Manslaughter will Caitlin be put with men or women? It depends on where Caitlin is charged. If Caitlin 
  MODEL: If Caitlin Jenner goes to prison for Vehicular Manslaughter will Caitlin be put with men or women? Based on the provided documents, there is no inform

  --- Example 2 ---
  HUMAN: In the middle of yawning, why did the music I was listening to change to a lower pitch? It changes the pressure of the cochlear fluid, which results i
  MODEL: In the middle of yawning, why did the music I was listening to change to a lower pitch? Based on the provided documents, there is no information expla

  --- Example 3 ---
  HUMAN: How do people with short term memory loss form long term mem

Featurizing p:   0%|          | 0/25 [00:00<?, ?it/s]

Tokenizing text...
Featurizing tokens


Featurizing q:   0%|          | 0/25 [00:00<?, ?it/s]

seed = 25
performing clustering in lower dimension = 147
kmeans time: 0.51 s
total discretization time: 1.17 seconds
  => MAUVE: 0.6211

Evaluating eli5-gemma-4-26b-a4b-it-cot_gemma26-shot1-ndoc3-42-quick_test200.json...
  n_items: 200

  --- Example 1 ---
  HUMAN: If Caitlin Jenner goes to prison for Vehicular Manslaughter will Caitlin be put with men or women? It depends on where Caitlin is charged. If Caitlin 
  MODEL: If Caitlin Jenner goes to prison for Vehicular Manslaughter will Caitlin be put with men or women? The provided documents do not contain information r

  --- Example 2 ---
  HUMAN: In the middle of yawning, why did the music I was listening to change to a lower pitch? It changes the pressure of the cochlear fluid, which results i
  MODEL: In the middle of yawning, why did the music I was listening to change to a lower pitch? Based on the provided documents, there is no information avail

  --- Example 3 ---
  HUMAN: How do people with short term memory loss form long 

Featurizing p:   0%|          | 0/25 [00:00<?, ?it/s]

Tokenizing text...
Featurizing tokens


Featurizing q:   0%|          | 0/25 [00:00<?, ?it/s]

seed = 25
performing clustering in lower dimension = 150
kmeans time: 0.53 s
total discretization time: 0.66 seconds
  => MAUVE: 1.082

Saved scores to /content/drive/MyDrive/mauve_scores_eli5.json
{
  "eli5-gemma-4-26b-a4b-it-openrouter-shot1-ndoc3-42-quick_test200.json": {
    "mauve": 0.6211
  },
  "eli5-gemma-4-26b-a4b-it-cot_gemma26-shot1-ndoc3-42-quick_test200.json": {
    "mauve": 1.082
  }
}
